# 04 深度学习实验

> 北京交通大学《机器学习与Python编程》研究性专题 — 裂纹图像识别系统
> 负责：王梓铭

使用自建的小型 CNN（CrackCNN，4 层卷积 + 全局平均池化）做裂纹二分类。
训练部分用了早停和 TensorBoard 记录，最后在测试集上看混淆矩阵和 ROC。
顺手跑了超参数网格搜索，看看 lr 和 dropout 哪个组合最好。

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_ROOT, DEVICE
from src.data_utils import apply_clahe, apply_median_filter, load_dataset, split_dataset
from src.evaluation.metrics import compute_metrics, plot_confusion_matrix, plot_roc_curve
from src.models.cnn import CrackCNN
from src.plot_config import set_chinese_font
from src.training.optimizers import grid_search_cnn
from src.training.trainer import CNNTrainer

set_chinese_font()
print(f"使用设备: {DEVICE}")

## 1. 数据加载与预处理

加载原始图像，统一缩放到 128×128，应用 CLAHE + 中值滤波增强。

In [ ]:
# ========== 超参数配置 ==========
IMAGE_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.5
PATIENCE = 10
MAX_SAMPLES = None  # None = 加载全部

print("配置：")
for k, v in [
    ("图像尺寸", IMAGE_SIZE), ("批次大小", BATCH_SIZE),
    ("训练轮数", EPOCHS), ("学习率", LEARNING_RATE),
    ("Dropout", DROPOUT_RATE), ("早停轮数", PATIENCE),
]:
    print(f"  {k}: {v}")

In [ ]:
# 加载 + 预处理
print("正在加载数据集...")
images_raw, labels = load_dataset(DATA_ROOT, max_samples=MAX_SAMPLES)
print(f"加载完成：{len(images_raw)} 张 ({np.sum(labels==1)} 正 / {np.sum(labels==0)} 负)")

print("正在预处理...")
images_processed = []
for img in tqdm(images_raw, desc="预处理"):
    enhanced = apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8))
    denoised = apply_median_filter(enhanced, kernel_size=5)
    resized = cv2.resize(denoised, (IMAGE_SIZE, IMAGE_SIZE))
    images_processed.append(resized)
images_processed = np.array(images_processed, dtype=np.float32)
print(f"预处理完成：{images_processed.shape}")

In [ ]:
# 划分 + 归一化 + DataLoader
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    images_processed, labels, train_ratio=0.7, val_ratio=0.15, random_seed=42
)
print(f"训练: {len(X_train)} | 验证: {len(X_val)} | 测试: {len(X_test)}")

# (N, H, W) → (N, 1, H, W)，归一化到 [0, 1]
def to_tensor(arr):
    return torch.from_numpy(arr[:, np.newaxis, :, :] / 255.0)

train_dataset = TensorDataset(to_tensor(X_train), torch.from_numpy(y_train).long())
val_dataset = TensorDataset(to_tensor(X_val), torch.from_numpy(y_val).long())
test_dataset = TensorDataset(to_tensor(X_test), torch.from_numpy(y_test).long())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"DataLoader 就绪")

## 2. CrackCNN 模型构建

In [ ]:
model = CrackCNN(num_classes=2, input_channels=1, dropout_rate=DROPOUT_RATE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"参数量: {total_params:,}")

# 验证前向传播
dummy = torch.randn(4, 1, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"输入: {dummy.shape} → 输出: {out.shape}")

## 3. 模型训练

In [ ]:
trainer = CNNTrainer(model, train_loader, val_loader, DEVICE, save_dir=PROJECT_ROOT / "outputs")
history = trainer.fit(
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    log_dir=str(PROJECT_ROOT / "outputs" / "tensorboard"),
)

## 4. 训练过程可视化

In [ ]:
fig = trainer.plot_history()
plt.show()

## 5. 测试集评估

In [ ]:
test_preds, test_probs, test_targets = trainer.evaluate(test_loader)
metrics = compute_metrics(test_targets, test_preds, test_probs)

print("测试集指标：")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")
print()
print(classification_report(test_targets, test_preds, target_names=["负样本", "正样本"], digits=4))

In [ ]:
fig_cm = plot_confusion_matrix(test_targets, test_preds)
plt.show()

In [ ]:
fig_roc = plot_roc_curve(test_targets, test_probs, label="CrackCNN")
plt.show()

## 6. 超参数网格搜索

快速实验不同学习率和 Dropout 组合。

In [ ]:
df_search = grid_search_cnn(
    train_loader, val_loader, DEVICE,
    lr_list=[1e-4, 5e-4, 1e-3],
    dropout_list=[0.3, 0.5, 0.7],
    epochs_per_trial=15,
    patience=5,
)
print("\n超参数搜索结果：")
print(df_search.to_string(index=False))

In [ ]:
# 可视化超参数敏感性
pivot = df_search.pivot_table(values="best_val_acc", index="lr", columns="dropout")
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels([f"{c:.1f}" for c in pivot.columns])
ax.set_yticklabels([f"{r:.0e}" for r in pivot.index])
ax.set_xlabel("Dropout")
ax.set_ylabel("Learning Rate")
ax.set_title("超参数敏感性 (验证准确率)")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.4f}", ha="center", va="center",
                fontweight="bold",
                color="white" if pivot.values[i, j] < pivot.values.max() * 0.7 else "black")
plt.colorbar(im, ax=ax, label="验证准确率")
plt.tight_layout()
plt.show()

## 7. 小结

### 模型设计要点

| 要素 | 说明 |
|------|------|
| 网络结构 | 4 Conv Block（双 Conv3×3 + BN + ReLU + MaxPool） + GAP + FC |
| 参数量 | ~1.17M |
| 正则化 | BatchNorm + Dropout |
| 池化 | Global Average Pooling，自适应输入尺寸 |
| 训练策略 | Adam + ReduceLROnPlateau + 早停 |

### 关键发现

- CrackCNN 端到端学习，无需手工特征提取
- 与 Notebook 02（传统 ML）和 Notebook 03（无监督聚类）的对比将在 Notebook 05 中统一展示